# 1D-CNN Seizure Prediction
Cross-patient benchmarking on CHB-MIT · 20 seeds · mean ± std

In [ ]:
from pathlib import Path
import sys

def find_repo_root(start):
    for path in [start, *start.parents]:
        if (path / 'src').exists() and (path / 'README.md').exists():
            return path
    return start

ROOT = find_repo_root(Path.cwd())
sys.path.insert(0, str(ROOT / 'src'))


In [ ]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW
from sklearn.metrics import roc_auc_score
from data_utils_leaky import get_leaky_dataloaders_for_seed as get_dataloaders_for_seed
from data_utils import SEEDS, DATA_DIR
from eval_utils import find_youden_threshold, full_evaluate

DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_CHANNELS   = 18
WIN          = 20 * 256   # 5120
BATCH_SIZE   = 128
MAX_EPOCHS   = 100
PATIENCE     = 20
LR           = 1e-4
WEIGHT_DECAY = 1e-4
OUT_DIR      = r'D:\seizure_results\1dcnn_leaky'

print(f'Device : {DEVICE}')
print(f'Seeds  : {len(SEEDS)}')
os.makedirs(OUT_DIR, exist_ok=True)

## Model Definition

In [ ]:
class MultiScaleBlock(nn.Module):
    """Parallel 1-D convolutions with three kernel sizes, concatenated."""
    def __init__(self, in_ch, out_ch, kernels=(3, 5, 7)):
        super().__init__()
        assert out_ch % len(kernels) == 0
        branch_ch = out_ch // len(kernels)
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(in_ch, branch_ch, k, padding=k // 2, bias=False),
                nn.BatchNorm1d(branch_ch),
                nn.GELU(),
            )
            for k in kernels
        ])

    def forward(self, x):
        return torch.cat([b(x) for b in self.branches], dim=1)


class CNN1D(nn.Module):
    """
    Three-stage 1-D CNN with multi-scale temporal convolutions.
    Input  : (B, 18, 5120)
    Output : (B, 2)
    """
    def __init__(self, n_channels=N_CHANNELS, dropout=0.5):
        super().__init__()
        self.encoder = nn.Sequential(
            MultiScaleBlock(n_channels, 96),
            nn.MaxPool1d(4),
            nn.Dropout(dropout * 0.5),

            MultiScaleBlock(96, 192),
            nn.MaxPool1d(4),
            nn.Dropout(dropout * 0.5),

            MultiScaleBlock(192, 384),
            nn.AdaptiveAvgPool1d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(384, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 2),
        )

    def forward(self, x):
        return self.classifier(self.encoder(x))


# Quick shape check
dummy = torch.zeros(4, N_CHANNELS, WIN)
print('Output shape:', CNN1D()(dummy).shape)  # expect (4, 2)

## Training & Evaluation Helpers

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * len(y)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def collect_probs(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    for x, y in loader:
        logits = model(x.to(DEVICE))
        probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        all_probs.append(probs)
        all_labels.append(y.numpy())
    return np.concatenate(all_probs), np.concatenate(all_labels)
print('Helpers defined.')

## Training Loop — Single Seed

In [ ]:
def run_seed(seed):
    print(f"\n{'='*60}")
    print(f"  Seed {seed}")
    print(f"{'='*60}")

    (train_loader, val_loader, test_loader,
     n_train, n_val, n_test,
     n_pre, n_inter) = get_dataloaders_for_seed(seed, DATA_DIR, BATCH_SIZE)

    _bx, _by = next(iter(train_loader))
    _n1 = int(_by.sum())
    print(f"  Batch check : {_n1}/{len(_by)} pre-ictal ({100*_n1/len(_by):.0f}%)")
    del _bx, _by

    model     = CNN1D().to(DEVICE)
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=5, min_lr=1e-7)
    criterion = nn.CrossEntropyLoss()

    best_val_auc  = 0.0
    best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    patience_left = PATIENCE

    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
        val_probs, val_labels = collect_probs(model, val_loader)
        val_auc = roc_auc_score(val_labels, val_probs)
        scheduler.step(val_auc)
        current_lr = optimizer.param_groups[0]['lr']

        print(f"  Epoch {epoch:3d} | loss {train_loss:.4f} | "
              f"val AUC {val_auc:.4f} | lr {current_lr:.2e}")

        if val_auc > best_val_auc:
            best_val_auc  = val_auc
            best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_left = PATIENCE
        else:
            patience_left -= 1
            if patience_left == 0:
                print(f"  Early stopping at epoch {epoch}.")
                break

    model.load_state_dict(best_state)
    model.to(DEVICE)

    val_probs, val_labels = collect_probs(model, val_loader)
    threshold = find_youden_threshold(val_labels, val_probs)
    print(f"  Youden threshold (val): {threshold:.4f}")

    test_probs, test_labels = collect_probs(model, test_loader)
    m = full_evaluate(test_labels, test_probs, threshold, stride_s=300)

    print(f"\n  [Seed {seed}] TEST  AUC {m['auc']:.4f} | "
          f"Sen {m['sensitivity']:.4f} | Spe {m['specificity']:.4f} | "
          f"Prec {m['precision']:.4f} | F1 {m['f1']:.4f} | "
          f"FAR {m['far']:.3f}/h | "
          f"EvtSen {m['event_sensitivity']:.3f} ({m['n_events']} events)")

    torch.save(best_state, os.path.join(OUT_DIR, f'seed{seed}_best.pt'))

    return {
        'seed':               seed,
        'test_auc':           m['auc'],
        'test_sen':           m['sensitivity'],
        'test_spe':           m['specificity'],
        'test_precision':     m['precision'],
        'test_f1':            m['f1'],
        'far':                m['far'],
        'event_sensitivity':  m['event_sensitivity'],
        'n_events':           m['n_events'],
        'threshold':          m['threshold'],
        'best_val_auc':       best_val_auc,
    }

print('run_seed defined.')

## Run All 20 Seeds

In [ ]:
all_results = []
for s in [42]:
    result = run_seed(s)
    all_results.append(result)

results_path = os.path.join(OUT_DIR, 'results.json')
with open(results_path, 'w') as _f:
    json.dump(
        [{k: (int(v) if k in ('seed', 'n_events') else float(v))
          for k, v in r.items()}
         for r in all_results],
        _f, indent=2,
    )
print(f'\nAll results saved to {results_path}')

## Result summary

In [ ]:
metrics = ['test_auc', 'test_sen', 'test_spe', 'test_precision',
           'test_f1', 'far', 'event_sensitivity']
labels  = ['AUC', 'Sensitivity (win)', 'Specificity', 'Precision',
           'F1', 'FAR (/h)', 'Sensitivity (event)']

print(f'1D-CNN  Cross-Patient Results (mean ± std, n={len(all_results)} seeds)')
print('-' * 55)
for m, l in zip(metrics, labels):
    vals = np.array([r[m] for r in all_results])
    print(f'  {l:<22s}: {vals.mean():.4f} ± {vals.std(ddof=1):.4f}')